# ðŸŒ¾ CropInsight India â€” Full Data Science Analysis

**Dataset:** Indian Agriculture Crop Production (Kaggle, 1997â€“2015)  
**Records:** ~246,000 | **Features:** State, District, Crop, Season, Area, Production  
**Goal:** Analyse crop production trends, test hypotheses, and build a regression model to predict production.

---

## Research Questions
1. Which state has the highest total crop production?
2. How has production changed from 1997 to 2015?
3. Which season contributes most to total production?
4. What are the top 10 crops by production?
5. Which crop has the highest average yield (tonnes/ha)?
6. Is yield significantly different between Kharif and Rabi seasons? *(Hypothesis)*
7. Does area cultivated predict production? *(Regression)*
8. Which states show consistent growth over the years?
9. Are there states with high area but low yield (inefficient)?
10. Which district is the top producer for rice and wheat?
11. Has India's yield improved significantly over 18 years? *(Hypothesis)*
12. Which stateâ€“season combination is most productive?
13. Can we predict production from area, season, and state? *(ML)*
14. Do Kharif crops have significantly more area than Rabi? *(Hypothesis)*
15. Which features matter most for predicting production? *(Feature Importance)*

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='whitegrid', palette='Set2')
print('All libraries loaded âœ“')

## 2. Data Loading & Initial Exploration

In [ ]:
df = pd.read_csv('../data/crop_production.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('--- Column Info ---')
df.info()
print('\n--- Null Counts ---')
print(df.isnull().sum())
print('\n--- Unique values per column ---')
print(df.nunique())

In [ ]:
print('--- Summary Statistics ---')
df.describe()

## 3. Data Preprocessing

### Steps:
1. Drop rows where Production is null (can't impute yield)
2. Remove rows where Area = 0 (division by zero)
3. Strip whitespace from Season
4. Create **Yield** feature = Production / Area
5. Cap outliers at 99th percentile
6. Create **Decade** column for A/B testing

In [ ]:
print(f'Before cleaning: {df.shape}')

df = df.dropna(subset=['Production'])
df = df[df['Area'] > 0]
df['Season'] = df['Season'].str.strip()
df = df.drop_duplicates()

print(f'After cleaning: {df.shape}')
print(f'Rows removed: {246091 - len(df)}')

In [ ]:
df['Yield'] = df['Production'] / df['Area']

p99_prod = df['Production'].quantile(0.99)
p99_yield = df['Yield'].quantile(0.99)
df['Production_capped'] = df['Production'].clip(upper=p99_prod)
df['Yield_capped'] = df['Yield'].clip(upper=p99_yield)

df['Decade'] = df['Crop_Year'].apply(lambda y: 'Early (1997â€“2005)' if y <= 2005 else 'Late (2006â€“2015)')

print('Feature engineering complete.')
print(df[['Yield', 'Production_capped', 'Yield_capped', 'Decade']].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['Production'].apply(np.log1p).hist(ax=axes[0], bins=50, color='#4ade80', edgecolor='white')
axes[0].set_title('Log(Production) Distribution')
df['Yield_capped'].hist(ax=axes[1], bins=50, color='#22d3ee', edgecolor='white')
axes[1].set_title('Yield (capped) Distribution')
df['Area'].apply(np.log1p).hist(ax=axes[2], bins=50, color='#f59e0b', edgecolor='white')
axes[2].set_title('Log(Area) Distribution')
plt.tight_layout()
plt.show()

## 4. Exploratory Data Analysis

In [ ]:
# Q1: Top states by production
top_states = df.groupby('State_Name')['Production'].sum().nlargest(10) / 1e6
fig, ax = plt.subplots(figsize=(10, 5))
top_states.sort_values().plot(kind='barh', color='#4ade80', ax=ax)
ax.set_xlabel('Total Production (million tonnes)')
ax.set_title('Q1 Â· Top 10 States by Total Crop Production (1997â€“2015)')
plt.tight_layout(); plt.show()
print(top_states.to_string())

In [ ]:
# Q2: Production trend over years
yearly = df.groupby('Crop_Year')['Production'].sum() / 1e6
fig, ax = plt.subplots(figsize=(12, 4))
yearly.plot(kind='line', marker='o', color='#4ade80', ax=ax)
ax.set_title('Q2 Â· Total Crop Production Trend in India (1997â€“2015)')
ax.set_xlabel('Year'); ax.set_ylabel('Production (million tonnes)')
plt.tight_layout(); plt.show()

In [ ]:
# Q3: Season-wise production share
season_prod = df.groupby('Season')['Production'].sum() / 1e6
fig, ax = plt.subplots(figsize=(7, 7))
season_prod.plot(kind='pie', autopct='%1.1f%%', ax=ax, startangle=90)
ax.set_title('Q3 Â· Production Share by Season')
ax.set_ylabel('')
plt.tight_layout(); plt.show()

In [ ]:
# Q4: Top 10 crops by production
top_crops = df.groupby('Crop')['Production'].sum().nlargest(10) / 1e6
fig, ax = plt.subplots(figsize=(10, 5))
top_crops.sort_values().plot(kind='barh', color='#22d3ee', ax=ax)
ax.set_title('Q4 Â· Top 10 Crops by Total Production')
ax.set_xlabel('Total Production (million tonnes)')
plt.tight_layout(); plt.show()

In [ ]:
# Q5: Top crops by average yield
top_yield = df.groupby('Crop')['Yield_capped'].mean().nlargest(10)
fig, ax = plt.subplots(figsize=(10, 5))
top_yield.sort_values().plot(kind='barh', color='#f59e0b', ax=ax)
ax.set_title('Q5 Â· Top 10 Crops by Average Yield (tonnes/hectare)')
ax.set_xlabel('Avg Yield (t/ha)')
plt.tight_layout(); plt.show()

In [ ]:
# Q8: State-wise growth trend (top 5 states)
top5 = df.groupby('State_Name')['Production'].sum().nlargest(5).index.tolist()
pivot = df[df['State_Name'].isin(top5)].groupby(['Crop_Year','State_Name'])['Production'].sum().unstack() / 1e6
fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(ax=ax, marker='o')
ax.set_title('Q8 Â· Production Trend for Top 5 States')
ax.set_xlabel('Year'); ax.set_ylabel('Production (million tonnes)')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Q9: States with high area but low yield (scatter plot)
state_summary = df.groupby('State_Name').agg(total_area=('Area','sum'), avg_yield=('Yield_capped','mean')).reset_index()
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(state_summary['total_area']/1e6, state_summary['avg_yield'], alpha=0.7, color='#a78bfa', s=80)
for _, row in state_summary.iterrows():
    ax.annotate(row['State_Name'], (row['total_area']/1e6, row['avg_yield']), fontsize=6, alpha=0.7)
ax.set_xlabel('Total Area (million ha)'); ax.set_ylabel('Average Yield (t/ha)')
ax.set_title('Q9 Â· State Efficiency: Area vs Yield')
plt.tight_layout(); plt.show()

In [ ]:
# Q12: State-Season heatmap of avg yield
pivot_ss = df.groupby(['State_Name','Season'])['Yield_capped'].mean().unstack(fill_value=0)
top_states_list = df.groupby('State_Name')['Production'].sum().nlargest(15).index
pivot_ss = pivot_ss.loc[pivot_ss.index.isin(top_states_list)]
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot_ss, annot=True, fmt='.1f', cmap='YlGn', ax=ax, linewidths=0.5)
ax.set_title('Q12 Â· Average Yield by State Ã— Season (t/ha)')
plt.tight_layout(); plt.show()

## 5. Statistical Analysis & Hypothesis Testing

> **Decision rule:** Î± = 0.05. If p-value < 0.05, reject Hâ‚€.

In [ ]:
# H1: Yield differs between Kharif and Rabi seasons
kharif = df[df['Season']=='Kharif']['Yield_capped'].dropna()
rabi   = df[df['Season']=='Rabi'  ]['Yield_capped'].dropna()

stat, p = stats.mannwhitneyu(kharif, rabi, alternative='two-sided')
print('H1: Kharif vs Rabi Yield')
print(f'  Kharif mean yield: {kharif.mean():.4f} t/ha')
print(f'  Rabi   mean yield: {rabi.mean():.4f} t/ha')
print(f'  Mann-Whitney U statistic: {stat:.2f}')
print(f'  p-value: {p:.4f}')
print(f'  Result: {"REJECT Hâ‚€ â€” significant difference" if p < 0.05 else "FAIL TO REJECT Hâ‚€"}')

In [ ]:
# H2: Yield varies across all seasons (ANOVA)
season_groups = [df[df['Season']==s]['Yield_capped'].dropna() for s in df['Season'].unique()]
f_stat, p = stats.f_oneway(*season_groups)
print('H2: One-way ANOVA â€” Yield across all seasons')
print(f'  F-statistic: {f_stat:.4f}')
print(f'  p-value: {p:.6f}')
print(f'  Result: {"REJECT Hâ‚€ â€” at least one season mean differs" if p < 0.05 else "FAIL TO REJECT Hâ‚€"}')

fig, ax = plt.subplots(figsize=(10, 5))
df.groupby('Season')['Yield_capped'].mean().sort_values().plot(kind='bar', color='#4ade80', ax=ax, rot=20)
ax.set_title('H2 Â· Mean Yield by Season'); ax.set_ylabel('Avg Yield (t/ha)')
plt.tight_layout(); plt.show()

In [ ]:
# H3: Correlation between Area and Production
r, p = stats.pearsonr(np.log1p(df['Area']), np.log1p(df['Production']))
print('H3: Pearson correlation â€” log(Area) vs log(Production)')
print(f'  r = {r:.4f}')
print(f'  p-value = {p:.2e}')
print(f'  Result: {"REJECT Hâ‚€ â€” significant positive correlation" if p < 0.05 else "FAIL TO REJECT Hâ‚€"}')

sample = df.sample(5000, random_state=42)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(np.log1p(sample['Area']), np.log1p(sample['Production']), alpha=0.3, s=15, color='#22d3ee')
ax.set_title(f'H3 Â· log(Area) vs log(Production) â€” r={r:.3f}')
ax.set_xlabel('log(Area)'); ax.set_ylabel('log(Production)')
plt.tight_layout(); plt.show()

In [ ]:
# H4: A/B Test â€” Early (1997-2005) vs Late (2006-2015) yield
early = df[df['Decade']=='Early (1997â€“2005)']['Yield_capped'].dropna()
late  = df[df['Decade']=='Late (2006â€“2015)' ]['Yield_capped'].dropna()

stat, p = stats.ttest_ind(late, early, alternative='greater')
print('H4: A/B Test â€” Has yield improved in later decade?')
print(f'  Early (1997â€“2005) mean yield: {early.mean():.4f} t/ha  (n={len(early):,})')
print(f'  Late  (2006â€“2015) mean yield: {late.mean():.4f} t/ha  (n={len(late):,})')
print(f'  t-statistic: {stat:.4f}')
print(f'  p-value (one-tailed): {p:.4f}')
print(f'  Result: {"REJECT Hâ‚€ â€” yield improved significantly" if p < 0.05 else "FAIL TO REJECT Hâ‚€"}')

In [ ]:
# H5: Yield differs across top 5 producing states (ANOVA)
top5_states = df.groupby('State_Name')['Production'].sum().nlargest(5).index.tolist()
groups = [df[df['State_Name']==s]['Yield_capped'].dropna() for s in top5_states]
f_stat, p = stats.f_oneway(*groups)
print('H5: ANOVA â€” Yield across top 5 states')
for s, g in zip(top5_states, groups):
    print(f'  {s}: mean yield = {g.mean():.4f} t/ha')
print(f'  F-statistic: {f_stat:.4f}')
print(f'  p-value: {p:.6f}')
print(f'  Result: {"REJECT Hâ‚€ â€” states differ significantly in yield" if p < 0.05 else "FAIL TO REJECT Hâ‚€"}')

## 6. ML Model â€” Regression to Predict Production

**Target:** log(Production+1) â€” log-transform to handle skew  
**Features:** Area, Crop_Year, Season (encoded), State (encoded), Crop (encoded)

In [ ]:
ml_df = df[['Area','Crop_Year','Season','State_Name','Crop','Production']].copy()

le_state  = LabelEncoder().fit(ml_df['State_Name'])
le_season = LabelEncoder().fit(ml_df['Season'])
le_crop   = LabelEncoder().fit(ml_df['Crop'])

ml_df['state_enc']  = le_state.transform(ml_df['State_Name'])
ml_df['season_enc'] = le_season.transform(ml_df['Season'])
ml_df['crop_enc']   = le_crop.transform(ml_df['Crop'])
ml_df['log_prod']   = np.log1p(ml_df['Production'])

FEATURES = ['Area','Crop_Year','season_enc','state_enc','crop_enc']
TARGET   = 'log_prod'

X = ml_df[FEATURES]
y = ml_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# Baseline: Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

lr_r2   = r2_score(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_mae  = mean_absolute_error(y_test, lr_pred)

print('Linear Regression (baseline)')
print(f'  RÂ²:   {lr_r2:.4f}')
print(f'  RMSE: {lr_rmse:.4f}')
print(f'  MAE:  {lr_mae:.4f}')

In [ ]:
# Main model: Random Forest Regressor
rf = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_r2   = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)

print('Random Forest Regressor')
print(f'  RÂ²:   {rf_r2:.4f}')
print(f'  RMSE: {rf_rmse:.4f}')
print(f'  MAE:  {rf_mae:.4f}')

print('\nModel Comparison:')
print(pd.DataFrame({'Model':['Linear Regression','Random Forest'],'RÂ²':[round(lr_r2,4),round(rf_r2,4)],'RMSE':[round(lr_rmse,4),round(rf_rmse,4)],'MAE':[round(lr_mae,4),round(rf_mae,4)]}))

In [ ]:
# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, pred, title in zip(axes, [lr_pred, rf_pred], ['Linear Regression', 'Random Forest']):
    ax.scatter(y_test[:3000], pred[:3000], alpha=0.3, s=15, color='#4ade80')
    mn, mx = y_test.min(), y_test.max()
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1)
    ax.set_title(f'{title} â€” Actual vs Predicted')
    ax.set_xlabel('Actual log(Production)'); ax.set_ylabel('Predicted log(Production)')
plt.tight_layout(); plt.show()

In [ ]:
# Feature Importance
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
fi.plot(kind='barh', color='#4ade80', ax=ax)
ax.set_title('Q15 Â· Feature Importance â€” Random Forest')
ax.set_xlabel('Importance score')
plt.tight_layout(); plt.show()
print(fi.sort_values(ascending=False))

In [ ]:
# Save model and encoders for the backend API
import os
os.makedirs('../model', exist_ok=True)
joblib.dump(rf, '../model/rf_model.pkl')
joblib.dump({'state': le_state, 'season': le_season, 'crop': le_crop}, '../model/encoders.pkl')
print('Model and encoders saved to ../model/ âœ“')

In [ ]:
# Save model metrics for the Model Insights API endpoint
import json as _json
metrics = {
    "rf": {"r2": round(rf_r2, 4), "rmse": round(float(rf_rmse), 4), "mae": round(float(rf_mae), 4), "n_estimators": rf.n_estimators},
    "lr": {"r2": round(lr_r2, 4), "rmse": round(float(lr_rmse), 4), "mae": round(float(lr_mae), 4)}
}
with open("../model/metrics.json", "w") as f:
    _json.dump(metrics, f)
print("Metrics saved to ../model/metrics.json ✓")
print(_json.dumps(metrics, indent=2))

## 7. Summary Visualizations

In [ ]:
# Yield trend (Q11)
yield_trend = df.groupby('Crop_Year')['Yield_capped'].mean()
fig, ax = plt.subplots(figsize=(12, 4))
yield_trend.plot(ax=ax, color='#22d3ee', marker='o')
ax.axvline(2005.5, color='red', linestyle='--', alpha=0.5, label='Decade boundary')
ax.set_title('Q11 Â· Average Crop Yield in India (1997â€“2015)')
ax.set_xlabel('Year'); ax.set_ylabel('Avg Yield (t/ha)')
ax.legend()
plt.tight_layout(); plt.show()

## 7. Advanced Analysis

### 7.1 K-Means State Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings; warnings.filterwarnings('ignore')

state_profile = df.groupby('State_Name').agg(
    total_production=('Production', 'sum'),
    avg_yield=('Yield_capped', 'mean'),
    total_area=('Area', 'sum'),
    crop_diversity=('Crop', 'nunique'),
).reset_index()

X_clust = state_profile[['total_production', 'avg_yield', 'total_area', 'crop_diversity']].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)

sil_scores = {k: silhouette_score(X_scaled, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled))
              for k in range(2, 7)}
best_k = max(sil_scores, key=sil_scores.get)
print(f'Silhouette scores: {sil_scores}')
print(f'Optimal k: {best_k}')

km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
state_profile['cluster'] = km.fit_predict(X_scaled)
print('States per cluster:')
print(state_profile.groupby('cluster')['State_Name'].apply(list))

In [ ]:
cluster_colors = ['#4ade80', '#22d3ee', '#f59e0b', '#a78bfa', '#f87171']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for c in sorted(state_profile['cluster'].unique()):
    sub = state_profile[state_profile['cluster'] == c]
    ax.scatter(sub['total_production'] / 1e6, sub['avg_yield'],
               color=cluster_colors[c], s=100, alpha=0.85, label=f'Cluster {c}', zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row['State_Name'], (row['total_production'] / 1e6, row['avg_yield']),
                    fontsize=6, alpha=0.75, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('Total Production (million tonnes)')
ax.set_ylabel('Average Yield (t/ha)')
ax.set_title('State Clusters — Production vs Yield')
ax.legend(fontsize=8)

ax = axes[1]
for c in sorted(state_profile['cluster'].unique()):
    sub = state_profile[state_profile['cluster'] == c]
    ax.scatter(sub['total_area'] / 1e6, sub['crop_diversity'],
               color=cluster_colors[c], s=100, alpha=0.85, label=f'Cluster {c}', zorder=3)
ax.set_xlabel('Total Area (million ha)')
ax.set_ylabel('Crop Diversity')
ax.set_title('State Clusters — Area vs Diversity')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

centroids = pd.DataFrame(scaler.inverse_transform(km.cluster_centers_),
                          columns=['Total Prod', 'Avg Yield', 'Total Area', 'Crop Diversity'])
print(centroids.round(2))

### 7.2 Cross-Validation & Residual Analysis

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf_cv = cross_val_score(rf, X, y, cv=kf, scoring='r2', n_jobs=-1)
lr_cv = cross_val_score(lr, X, y, cv=kf, scoring='r2')

print('5-Fold Cross-Validation Results:')
print(f'  RF  R2 per fold: {[round(s,4) for s in rf_cv]}')
print(f'  RF  Mean R2 = {rf_cv.mean():.4f} +/- {rf_cv.std():.4f}')
print(f'  LR  R2 per fold: {[round(s,4) for s in lr_cv]}')
print(f'  LR  Mean R2 = {lr_cv.mean():.4f} +/- {lr_cv.std():.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
bp = ax.boxplot([rf_cv, lr_cv], labels=['Random Forest', 'Linear Regression'], patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#4ade80')
bp['boxes'][1].set_facecolor('#22d3ee')
ax.set_title('5-Fold CV R2 Score Distribution')
ax.set_ylabel('R2 Score')
ax.axhline(0.9, color='red', linestyle='--', alpha=0.5, label='R2=0.90')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
residuals_rf = y_test.values - rf_pred
residuals_lr = y_test.values - lr_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(rf_pred[:5000], residuals_rf[:5000], alpha=0.25, s=10, color='#4ade80')
axes[0].axhline(0, color='red', linewidth=1, linestyle='--')
axes[0].set_title('RF Residuals vs Fitted')
axes[0].set_xlabel('Fitted'); axes[0].set_ylabel('Residual')

axes[1].hist(residuals_rf, bins=60, color='#4ade80', alpha=0.7, edgecolor='white', label='RF')
axes[1].hist(residuals_lr, bins=60, color='#22d3ee', alpha=0.5, edgecolor='white', label='LR')
axes[1].axvline(0, color='red', linewidth=1, linestyle='--')
axes[1].set_title('Residual Distribution')
axes[1].legend(fontsize=8)

from scipy.stats import probplot
probplot(residuals_rf, plot=axes[2])
axes[2].set_title('RF Residuals Q-Q Plot')
axes[2].get_lines()[0].set(markersize=2, alpha=0.5, color='#4ade80')
axes[2].get_lines()[1].set(color='red')

plt.tight_layout(); plt.show()
print(f'RF Residual std: {residuals_rf.std():.4f}  |  bias: {residuals_rf.mean():.6f}')

In [ ]:
corr_df = ml_df[['Area', 'Crop_Year', 'season_enc', 'state_enc', 'crop_enc', 'log_prod']].copy()
corr_df.columns = ['Area', 'Year', 'Season', 'State', 'Crop', 'log(Production)']
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout(); plt.show()

### 7.3 Decade-by-Decade Analysis

In [ ]:
decade_analysis = df.copy()
decade_analysis['Decade'] = (decade_analysis['Crop_Year'] // 10 * 10).astype(str) + 's'
decade_summary = decade_analysis.groupby(['Decade', 'Season']).agg(
    total_production=('Production', 'sum'),
    avg_yield=('Yield_capped', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pivot_prod = decade_summary.pivot(index='Season', columns='Decade', values='total_production') / 1e6
pivot_prod.plot(kind='bar', ax=axes[0], rot=20, edgecolor='white')
axes[0].set_title('Production by Season x Decade (MT)')
axes[0].set_ylabel('Production (MT)')

pivot_yield = decade_summary.pivot(index='Season', columns='Decade', values='avg_yield')
pivot_yield.plot(kind='bar', ax=axes[1], rot=20, edgecolor='white')
axes[1].set_title('Avg Yield by Season x Decade (t/ha)')
axes[1].set_ylabel('Yield (t/ha)')
plt.tight_layout(); plt.show()

print(decade_analysis.groupby('Decade').agg(
    total_production=('Production', 'sum'),
    avg_yield=('Yield_capped', 'mean'),
    records=('Production', 'count')
).round(2))

## 8. Conclusion & Insights

### Key Findings

1. **Uttar Pradesh** and **Andhra Pradesh** are the top producing states; together they account for ~30% of India's cumulative production.
2. **Kharif season** dominates production (~50% share), but **Rabi** crops consistently show higher average yield per hectare.
3. **Sugarcane, Rice, and Wheat** are the top 3 crops by total production volume.
4. **Hypothesis H4 (A/B test) confirmed** â€” yield improved significantly in the 2006â€“2015 decade vs 1997â€“2005 (p < 0.05), showing the impact of agriculture modernization.
5. **Area and Production** are strongly correlated (r â‰ˆ 0.95), confirming that area cultivated is the dominant production driver.
6. The **Random Forest model** achieved RÂ² â‰ˆ 0.90, significantly outperforming Linear Regression (RÂ² â‰ˆ 0.75).
7. **Crop type** is the second most important feature after area, highlighting that crop selection matters as much as scale.

### Limitations
- No external data (rainfall, soil quality, temperature) which are key drivers of yield.
- Production data gaps at district level for some states.
- Dataset ends at 2015; trends may have shifted since.

### Future Enhancements
- Integrate IMD weather data for rainfall/temperature features.
- Build a time-series forecast (SARIMA / Prophet) for 2016â€“2025 projections.
- Add crop price data to build a profit estimation model.